In [55]:
import os
import json
import torch
from datetime import datetime, timezone
import re


In [56]:
import torch

TRIAGE_BLACKBOARD_PATH = "/kaggle/usr/lib/notebooks/premshaw23/triage_agent/triage_blackboard.pt"

triage_blackboard = torch.load(
    TRIAGE_BLACKBOARD_PATH,
    map_location="cpu",
    weights_only=False
)

print("Triage blackboard loaded successfully.")
print(triage_blackboard)

Triage blackboard loaded successfully.
{'agent': 'TriageAgent', 'input_agent': 'ConsistencyCheckerAgent', 'output': {'agent': 'TriageAgent', 'grade': 0, 'grade_label': 'No DR', 'confidence': 0.36490851640701294, 'base_referral_tier': 'routine', 'access_adjusted_tier': 'routine', 'referral_tier': 'routine', 'referral_tier_label': 'ROUTINE', 'urgency_cues': {'vitreous_hemorrhage': False, 'sudden_vision_loss': False, 'rubeosis': False}, 'emergency_cues_detected': [], 'tta_spread': 0, 'tta_consistent': True, 'clinical_completeness': 'LARGELY_MISSING', 'clinical_missing_fields': ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms'], 'access_context_used': False, 'access_adjustment_reason': 'No access-context adjustment applied.', 'review_required': False, 'review_reason': None, 'triage_status': 'ROUTED'}, 'timestamp': '2026-09-26T07:20:09.734262+00:00'}


In [57]:
triage_output = triage_blackboard["output"]

print("Triage output loaded:")
print(triage_output)

Triage output loaded:
{'agent': 'TriageAgent', 'grade': 0, 'grade_label': 'No DR', 'confidence': 0.36490851640701294, 'base_referral_tier': 'routine', 'access_adjusted_tier': 'routine', 'referral_tier': 'routine', 'referral_tier_label': 'ROUTINE', 'urgency_cues': {'vitreous_hemorrhage': False, 'sudden_vision_loss': False, 'rubeosis': False}, 'emergency_cues_detected': [], 'tta_spread': 0, 'tta_consistent': True, 'clinical_completeness': 'LARGELY_MISSING', 'clinical_missing_fields': ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms'], 'access_context_used': False, 'access_adjustment_reason': 'No access-context adjustment applied.', 'review_required': False, 'review_reason': None, 'triage_status': 'ROUTED'}


In [58]:
import os
import torch

COMMON_IMAGE_BLACKBOARDS = [
    "/kaggle/usr/lib/notebooks/divyanshukj4495/imageagent/image_blackboard.pt",
    "/kaggle/usr/lib/notebooks/divyanshukj4495/imageagent/image_agent_full.pth",
]

COMMON_NOTE_BLACKBOARDS = [
    "/kaggle/usr/lib/notebooks/divyanshukj4495/clinical_note_agent/clinical_blackboard.pt",
]

def load_first_existing(paths):
    for path in paths:
        if os.path.exists(path):
            return torch.load(
                path,
                map_location="cpu",
                weights_only=False
            ), path
    return None, None

image_blackboard, image_blackboard_path = load_first_existing(
    COMMON_IMAGE_BLACKBOARDS
)

note_blackboard, note_blackboard_path = load_first_existing(
    COMMON_NOTE_BLACKBOARDS
)

print("Image blackboard:", image_blackboard_path)
print("Note blackboard:", note_blackboard_path)

Image blackboard: /kaggle/usr/lib/notebooks/divyanshukj4495/imageagent/image_blackboard.pt
Note blackboard: /kaggle/usr/lib/notebooks/divyanshukj4495/clinical_note_agent/clinical_blackboard.pt


In [59]:
def unwrap_output(blackboard):
    if not isinstance(blackboard, dict):
        return {}

    if isinstance(blackboard.get("output"), dict):
        return blackboard["output"]

    return blackboard

image_output = unwrap_output(image_blackboard)
note_output = unwrap_output(note_blackboard)

print("Image evidence keys:", list(image_output.keys()))
print("Note evidence keys:", list(note_output.keys()))

Image evidence keys: ['grade', 'confidence', 'image_embedding', 'gradcam', 'lesion_boxes', 'segmentation_mask', 'tta_grades', 'tta_confidences', 'grade_range', 'review_required']
Note evidence keys: ['image_id', 'visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'diabetes_duration', 'prior_vitrectomy', 'symptoms', 'image_quality', 'missing_fields']


In [60]:
EVIDENCE_CORPUS = [
    {
        "id": "RETINA_RULE_01",
        "source": "RetinaAgent system rule",
        "text": "Sudden vision loss or vitreous hemorrhage is treated as an emergency cue in the prototype triage policy."
    },
    {
        "id": "RETINA_RULE_02",
        "source": "RetinaAgent system rule",
        "text": "A TTA grade spread greater than one grade triggers clinician review."
    },
    {
        "id": "RETINA_RULE_03",
        "source": "RetinaAgent system rule",
        "text": "The referral tier is derived from the predicted diabetic retinopathy grade and urgency cues, with access constraints used for refinement."
    },
    {
        "id": "RETINA_RULE_04",
        "source": "RetinaAgent explanation policy",
        "text": "Explanation claims must be traceable to image localization evidence, clinical-note entities, or retrieved evidence."
    },
    {
        "id": "RETINA_RULE_05",
        "source": "RetinaAgent explanation policy",
        "text": "Missing clinical-note fields must not be fabricated and should remain explicitly marked as missing."
    },
]

def retrieve_evidence(query, top_k=5):
    """Lightweight offline retrieval for the MVP."""
    query_tokens = {
        word.lower().strip(".,:;()[]")
        for word in str(query).split()
        if len(word) > 2
    }

    scored = []

    for item in EVIDENCE_CORPUS:
        text_tokens = {
            word.lower().strip(".,:;()[]")
            for word in item["text"].split()
            if len(word) > 2
        }

        score = len(query_tokens & text_tokens)

        scored.append({
            **item,
            "retrieval_score": score
        })

    scored.sort(
        key=lambda x: x["retrieval_score"],
        reverse=True
    )

    return scored[:top_k]

In [61]:
GRADE_LABELS = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "PDR",
}

REFERRAL_LABELS = {
    "routine": "Routine monitoring",
    "soon": "Review soon",
    "urgent": "Urgent ophthalmology review",
    "emergency": "Emergency review",
    "clinician_review": "Clinician review required",
}

In [62]:
def extract_image_evidence(image_output):
    """
    Extract evidence that is actually produced by the Image Agent.
    Do not invent lesion findings from raw model outputs.
    """
    if not isinstance(image_output, dict):
        return {}

    evidence = {}

    # Classification evidence
    for key in [
        "grade",
        "confidence",
        "tta_grades",
        "tta_confidences",
        "grade_range",
        "review_required",
    ]:
        value = image_output.get(key)

        if value not in [None, "", [], {}]:
            evidence[key] = value

    # Localization / visual evidence
    for key in [
        "gradcam",
        "lesion_boxes",
        "segmentation_mask",
    ]:
        value = image_output.get(key)

        if value is not None:
            evidence[key] = value

    return evidence


def extract_note_evidence(note_output):
    """Extract only non-empty clinical-note entities."""
    evidence = {}

    if not isinstance(note_output, dict):
        return evidence

    allowed_keys = [
        "visual_acuity",
        "lens_status",
        "prior_laser",
        "prior_anti_vegf",
        "hba1c",
        "diabetes_duration",
        "prior_vitrectomy",
        "symptoms",
        "image_quality",
    ]

    for key in allowed_keys:
        value = note_output.get(key)

        if value not in [None, "", [], {}]:
            evidence[key] = value

    return evidence


image_evidence = extract_image_evidence(image_output)
note_evidence = extract_note_evidence(note_output)

print("Image evidence:")
print(json.dumps(image_evidence, indent=2, default=str))

print("\nClinical-note evidence:")
print(json.dumps(note_evidence, indent=2, default=str))

Image evidence:
{
  "grade": 0,
  "confidence": 0.36490851640701294,
  "tta_grades": [
    0,
    0,
    0
  ],
  "tta_confidences": [
    0.36490851640701294,
    0.33550402522087097,
    0.41503438353538513
  ],
  "grade_range": 0,
  "review_required": false,
  "gradcam": "[[0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n ...\n [0.         0.         0.         ... 0.         0.         0.        ]\n [0.         0.         0.         ... 0.         0.         0.        ]\n [0.         0.         0.         ... 0.         0.         0.        ]]",
  "lesion_boxes": [
    {
      "x": 0,
      "y": 0,
      "width": 111,
      "height": 99,
      "area": 7790
    },
    {
      "x": 51,
      "y": 0,
      "width": 173,
      "height": 224,
      "area": 15634
    },
    {
      "x": 0,
      "y": 129,
      "width"

In [63]:
def format_note_facts(note_evidence):
    facts = []

    if "diabetes_duration" in note_evidence:
        facts.append(
            f"diabetes duration: {note_evidence['diabetes_duration']} years"
        )

    if "hba1c" in note_evidence:
        facts.append(f"HbA1c: {note_evidence['hba1c']}")

    if "visual_acuity" in note_evidence:
        facts.append(f"visual acuity: {note_evidence['visual_acuity']}")

    if "lens_status" in note_evidence:
        facts.append(f"lens status: {note_evidence['lens_status']}")

    if "prior_laser" in note_evidence:
        facts.append(f"prior laser: {note_evidence['prior_laser']}")

    if "prior_anti_vegf" in note_evidence:
        facts.append(
            f"prior anti-VEGF: {note_evidence['prior_anti_vegf']}"
        )

    if "prior_vitrectomy" in note_evidence:
        facts.append(
            f"prior vitrectomy: {note_evidence['prior_vitrectomy']}"
        )

    if "symptoms" in note_evidence:
        facts.append(f"symptoms: {note_evidence['symptoms']}")

    if "image_quality" in note_evidence:
        facts.append(f"image quality: {note_evidence['image_quality']}")

    return facts


def format_image_facts(image_evidence):
    return [
        f"{key}: {value}"
        for key, value in image_evidence.items()
    ]

In [64]:
def build_explanation_evidence(
    triage_output,
    image_evidence,
    note_evidence
):
    query = (
        f"DR grade {triage_output['grade']} "
        f"{triage_output['grade_label']}, "
        f"referral tier {triage_output['referral_tier']}, "
        f"urgency cues "
        f"{triage_output.get('emergency_cues_detected', [])}, "
        f"TTA spread {triage_output.get('tta_spread')}"
    )

    retrieved = retrieve_evidence(query, top_k=5)

    return {
        "grade_evidence": {
            "source_type": "grader/consistency",
            "grade": triage_output["grade"],
            "grade_label": triage_output["grade_label"],
            "confidence": triage_output.get("confidence"),
        },
        "image_evidence": image_evidence,
        "clinical_note_evidence": note_evidence,
        "retrieved_evidence": retrieved,
    }

explanation_evidence = build_explanation_evidence(
    triage_output,
    image_evidence,
    note_evidence
)

print(json.dumps(explanation_evidence, indent=2, default=str))

{
  "grade_evidence": {
    "source_type": "grader/consistency",
    "grade": 0,
    "grade_label": "No DR",
    "confidence": 0.36490851640701294
  },
  "image_evidence": {
    "grade": 0,
    "confidence": 0.36490851640701294,
    "tta_grades": [
      0,
      0,
      0
    ],
    "tta_confidences": [
      0.36490851640701294,
      0.33550402522087097,
      0.41503438353538513
    ],
    "grade_range": 0,
    "review_required": false,
    "gradcam": "[[0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]\n ...\n [0.         0.         0.         ... 0.         0.         0.        ]\n [0.         0.         0.         ... 0.         0.         0.        ]\n [0.         0.         0.         ... 0.         0.         0.        ]]",
    "lesion_boxes": [
      {
        "x": 0,
        "y": 0,
        "width": 111,
  

In [65]:
def explanation_agent(
    triage_output,
    image_evidence,
    note_evidence,
    retrieved_evidence,
):
    grade = triage_output["grade"]
    grade_label = triage_output["grade_label"]
    confidence = triage_output.get("confidence")
    referral_tier = triage_output["referral_tier"]

    referral_label = triage_output.get(
        "referral_tier_label",
        REFERRAL_LABELS.get(referral_tier, referral_tier)
    )

    emergency_cues = triage_output.get(
        "emergency_cues_detected",
        []
    )

    tta_spread = triage_output.get("tta_spread")
    tta_consistent = triage_output.get("tta_consistent")

    image_facts = format_image_facts(image_evidence)
    note_facts = format_note_facts(note_evidence)

    # ---------------------------
    # Clinician-facing explanation
    # ---------------------------

    clinician_parts = []

    confidence_text = (
        f" with confidence {float(confidence):.2f}"
        if isinstance(confidence, (int, float))
        else ""
    )

    clinician_parts.append(
        f"The model assessment is {grade_label} (grade {grade})"
        f"{confidence_text}."
    )

    if image_facts:
        clinician_parts.append(
            "Image-localization evidence available to this explanation: "
            + "; ".join(image_facts)
            + "."
        )
    else:
        clinician_parts.append(
            "No structured lesion-localization finding was available "
            "from the loaded Image Agent blackboard, so no specific "
            "lesion is asserted."
        )

    if note_facts:
        clinician_parts.append(
            "Available clinical-note evidence: "
            + "; ".join(note_facts)
            + "."
        )
    else:
        clinician_parts.append(
            "No non-empty structured clinical-note entities were available; "
            "missing fields are not inferred."
        )

    if emergency_cues:
        clinician_parts.append(
            "The triage output contains the following emergency cue(s): "
            + ", ".join(emergency_cues)
            + "."
        )

    clinician_parts.append(
        f"The current triage output is {referral_label}."
    )

    if tta_spread is not None:
        clinician_parts.append(
            f"TTA grade spread is {tta_spread}; "
            f"consistency status is "
            f"{'consistent' if tta_consistent else 'not consistent'}."
        )

    clinician_parts.append(
        "This explanation is an evidence-grounded research prototype "
        "and is not a clinical diagnosis."
    )

    clinician_text = " ".join(clinician_parts)

    # ---------------------------
    # Patient-facing explanation
    # ---------------------------

    patient_parts = [
        f"The screening model classified the retinal image as {grade_label}."
    ]

    if referral_tier == "emergency":
        patient_parts.append(
            "The system detected an emergency-related cue and therefore "
            "flagged the case for emergency review."
        )
    elif referral_tier == "clinician_review":
        patient_parts.append(
            "The model output requires clinician review because the "
            "prediction consistency check was not satisfied."
        )
    else:
        patient_parts.append(
            f"The system assigned the current follow-up category as "
            f"{referral_label.lower()}."
        )

    if note_facts:
        patient_parts.append(
            "Some available clinical information was also considered."
        )

    patient_parts.append(
        "This is a screening-support output and should be reviewed by "
        "a qualified eye-care professional."
    )

    patient_text = " ".join(patient_parts)

    # Create structured claims from the actual generated text.
    # This prevents the downstream Verifier from interpreting numeric
    # arrays/tensors as textual claims.
    def split_text_into_claims(text):
        if not isinstance(text, str):
            return []

        text = re.sub(r"\s+", " ", text).strip()

        if not text:
            return []

        return [
            sentence.strip()
            for sentence in re.split(r"(?<=[.!?])\s+", text)
            if sentence.strip()
        ]

    claims = []

    for claim in split_text_into_claims(clinician_text):
        claims.append({
            "claim": claim,
            "output": "clinician"
        })

    for claim in split_text_into_claims(patient_text):
        claims.append({
            "claim": claim,
            "output": "patient"
        })

    return {
        "agent": "ExplanationAgent",
        "clinician_explanation": clinician_text,
        "patient_explanation": patient_text,

        # Structured textual claims for the Verifier Agent.
        "claims": claims,

        "evidence": {
            "image": image_evidence,
            "clinical_note": note_evidence,
            "retrieved": retrieved_evidence,
        },

        # Triage provenance is retained separately.
        # It is not substituted for L/e/R evidence.
        "triage_provenance": {
            "grade": grade,
            "grade_label": grade_label,
            "confidence": confidence,
            "referral_tier": referral_tier,
            "referral_label": referral_label,
            "emergency_cues": emergency_cues,
            "tta_spread": tta_spread,
            "tta_consistent": tta_consistent,
        },

        "grounding_policy": (
            "Claims are restricted to loaded model outputs, "
            "clinical-note entities, image-localization evidence, "
            "and retrieved evidence. "
            "Triage provenance is retained explicitly for referral-related "
            "claims and is not treated as image or clinical-note evidence."
        ),
    }

In [66]:
explanation_output = explanation_agent(
    triage_output=triage_output,
    image_evidence=image_evidence,
    note_evidence=note_evidence,
    retrieved_evidence=explanation_evidence["retrieved_evidence"],
)

print("CLINICIAN-FACING EXPLANATION")
print("=" * 70)
print(explanation_output["clinician_explanation"])

print("\nPATIENT-FACING EXPLANATION")
print("=" * 70)
print(explanation_output["patient_explanation"])

CLINICIAN-FACING EXPLANATION
The model assessment is No DR (grade 0) with confidence 0.36. Image-localization evidence available to this explanation: grade: 0; confidence: 0.36490851640701294; tta_grades: [0, 0, 0]; tta_confidences: [0.36490851640701294, 0.33550402522087097, 0.41503438353538513]; grade_range: 0; review_required: False; gradcam: [[0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]
 [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]
 [0.02140775 0.02140775 0.02140775 ... 0.03441945 0.03441945 0.03441945]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]; lesion_boxes: [{'x': 0, 'y': 0, 'width': 111, 'height': 99, 'area': 7790}, {'x': 51, 'y': 0, 'width': 173, 'height': 224, 'area': 15634}, {'x': 0, 'y': 129, 'width': 44, 'height': 46, 'area': 1495}, {'x': 45,

In [67]:
explanation_blackboard = {
    "agent": "ExplanationAgent",
    "input_agent": "TriageAgent",
    "output": explanation_output,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

EXPLANATION_BLACKBOARD_PATH = (
    "/kaggle/working/explanation_blackboard.pt"
)

torch.save(
    explanation_blackboard,
    EXPLANATION_BLACKBOARD_PATH
)

print(
    f"Saved Explanation Agent blackboard to: "
    f"{EXPLANATION_BLACKBOARD_PATH}"
)

Saved Explanation Agent blackboard to: /kaggle/working/explanation_blackboard.pt


In [68]:
required_keys = [
    "agent",
    "clinician_explanation",
    "patient_explanation",
    "claims",
    "evidence",
    "grounding_policy",
]

missing = [
    key for key in required_keys
    if key not in explanation_output
]

assert not missing, f"Missing explanation keys: {missing}"

assert isinstance(
    explanation_output["clinician_explanation"],
    str
)

assert isinstance(
    explanation_output["patient_explanation"],
    str
)

assert isinstance(
    explanation_output["claims"],
    list
)

for item in explanation_output["claims"]:
    assert isinstance(item, dict)
    assert item.get("output") in ["clinician", "patient"]
    assert isinstance(item.get("claim"), str)
    assert item["claim"].strip()

assert "image" in explanation_output["evidence"]
assert "clinical_note" in explanation_output["evidence"]
assert "retrieved" in explanation_output["evidence"]

print("Explanation Agent validation: PASSED")
print("Structured claims:", len(explanation_output["claims"]))
print("Blackboard ready for Verifier Agent.")


Explanation Agent validation: PASSED
Structured claims: 42
Blackboard ready for Verifier Agent.
